In [1]:
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import networkx as nx
import sspa

In [2]:
# Get reactome pathways and exported to gmt file ### DON'T RUN THIS LINE IF YOU HAVE THE FILE
reactome_pathways = sspa.process_reactome(organism="Homo sapiens", download_latest=True, filepath='.')

Beginning Reactome download...
Reactome DB file saved to ./Reactome_Homo_sapiens_pathways_ChEBI_R90.gmt
Complete!


In [3]:
reactome_pathways

,Pathway_name,0,1,2,3,4,5,6,7,8,...,1540,1541,1542,1543,1544,1545,1546,1547,1548,1549
R-HSA-1059683,Interleukin-6 signaling,30616,456216,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
R-HSA-109581,Apoptosis,61120,4705,456216,28494,36080,15377,29105,43474,47575,...,None,None,None,None,None,None,None,None,None,None
R-HSA-109582,Hemostasis,15366,91144,15377,15378,15379,456215,456216,35366,57895,...,None,None,None,None,None,None,None,None,None,None
R-HSA-109606,Intrinsic Pathway for Apoptosis,456216,28494,36080,15377,43474,47575,30616,None,None,...,None,None,None,None,None,None,None,None,None,None
R-HSA-109703,PKB-mediated events,456216,57836,15377,58165,456215,30616,None,None,None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
R-HSA-9918443,Defective visual phototransduction due to OPN1...,16066,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
R-HSA-9918449,Defective visual phototransduction due to STRA...,17336,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
R-HSA-9918450,Defective visual phototransduction due to OPN1...,16066,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
R-HSA-9918454,Defective visual phototransduction due to ABCA...,30616,15377,71063,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [4]:
reactome_pathways = pd.read_csv('Reactome_Homo_sapiens_pathways_ChEBI_R89.gmt',delimiter='\t',header=None)

In [16]:
reactome_pathways = reactome_pathways.rename(columns={1: 'Pathway_name'})
reactome_pathways = reactome_pathways.set_index(reactome_pathways.columns[0])

In [17]:
reactome_pathways

,Pathway_name,2,3,4,5,6,7,8,9,10,...,1543,1544,1545,1546,1547,1548,1549,1550,1551,1552
0,,,,,,,,,,,,,,,,,,,,,
R-HSA-1059683,Interleukin-6 signaling,30616,456216.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
R-HSA-109581,Apoptosis,61120,4705.0,456216.0,28494.0,36080.0,15377.0,29105.0,43474.0,47575.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
R-HSA-109582,Hemostasis,15366,91144.0,15377.0,15378.0,15379.0,456215.0,456216.0,35366.0,57895.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
R-HSA-109606,Intrinsic Pathway for Apoptosis,456216,28494.0,36080.0,15377.0,43474.0,47575.0,30616.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
R-HSA-109703,PKB-mediated events,456216,57836.0,15377.0,58165.0,456215.0,30616.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
R-HSA-9861718,Regulation of pyruvate metabolism,15361,17154.0,189572.0,229639.0,15377.0,15378.0,456216.0,30616.0,29103.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
R-HSA-9864848,Complex IV assembly,29036,57613.0,28494.0,49552.0,29105.0,23378.0,61715.0,18420.0,83282.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
R-HSA-9865881,Complex III assembly,49601,33739.0,36080.0,26355.0,61717.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
chebi_identified = pd.read_csv('cleaned_chebi_list.txt',header=None)
chebi_identified_lpa = pd.read_csv('/home/scostagonza/Documents/chebis_lpa_estelle_values.txt',header=None)

In [43]:
chebi_identified_lpa = pd.unique(chebi_identified_lpa[0])

In [18]:
def find_root(G,child):
    parent = list(G.predecessors(child))
    if len(parent) == 0:
        return child
    else:  
        return find_root(G, parent[0])
    
# load the pathway database file from the data folder

hierarchy = pd.read_csv('/home/scostagonza/Documents/Pathway_hierarchy_rel.txt', sep='\t', header=None)
hierarchy_hsa = hierarchy[hierarchy[0].str.contains('HSA')]
hierarchy_hsa_parents = np.setdiff1d(hierarchy_hsa[0], hierarchy_hsa[1])
hierarchy_hsa_all = pd.concat([hierarchy_hsa, pd.DataFrame([hierarchy_hsa_parents, hierarchy_hsa_parents], index=[0, 1]).T])

# the default graph is the pathway hierarchy coloured by root pathway membership as defined by Reactome
G = nx.from_pandas_edgelist(hierarchy_hsa, source=0, target=1, create_using=nx.DiGraph())

In [19]:
# the default graph is the pathway hierarchy coloured by root pathway membership as defined by Reactome
G = nx.from_pandas_edgelist(hierarchy_hsa, source=0, target=1, create_using=nx.DiGraph())
hierarchy_hsa_all['Root'] = [find_root(G, i) for i in hierarchy_hsa_all[1]]
root_cmap = dict(zip(set(hierarchy_hsa_all['Root']), sns.color_palette("husl", len(set(hierarchy_hsa_all['Root']))).as_hex()))

In [20]:
name_dict = dict(zip(reactome_pathways.index, reactome_pathways['Pathway_name']))
G.add_nodes_from([(node, {'Name': attr, 'label': attr}) for (node, attr) in name_dict.items()])

In [21]:
G.add_nodes_from([(node, {'Root': attr, 
                              'RootCol': root_cmap[attr], 
                              'color': root_cmap[attr], 
                              'RootName': name_dict[attr]}) for (node, attr) in dict(zip(hierarchy_hsa_all[1], hierarchy_hsa_all['Root'])).items()])

In [22]:
df = reactome_pathways.iloc[:,2:].fillna(0)
df = df.astype(int)

In [54]:
# match_counts = df.apply(lambda row: row.isin(chebi_identified[0].values).sum(), axis=1)
# count_greater_than_zero = lambda row: (row > 0).sum()
# counts = df.apply(count_greater_than_zero, axis=1)
# coverage = match_counts / counts

In [50]:
match_counts = df.apply(lambda row: row.isin(chebi_identified_lpa).sum(), axis=1)
count_greater_than_zero = lambda row: (row > 0).sum()
counts = df.apply(count_greater_than_zero, axis=1)
coverage = match_counts / counts

In [64]:
coverage

R-HSA-1059683    0.000000
R-HSA-109581     0.000000
R-HSA-109582     0.035294
R-HSA-109606     0.000000
R-HSA-109703     0.000000
                   ...   
R-HSA-9918443         NaN
R-HSA-9918449         NaN
R-HSA-9918450         NaN
R-HSA-9918454    0.000000
R-HSA-997272     0.000000
Length: 2394, dtype: float64

In [65]:
for node in G.nodes:
    if node in coverage.index:
        G.nodes[node]['coverage'] = coverage[node]
    else:
        G.nodes[node]['coverage'] = 0  

In [66]:
nx.write_graphml(G, '/home/scostagonza/Documents/Pathway_hierarchy.graphml')

In [67]:
df = pd.read_csv('Pathway_coverage.csv')
#df['coverage'] = df['#Entities found']/df['#Entities total']
#df.coverage

In [68]:
df

,Pathway identifier,Pathway name,#Entities found,#Entities total,Entities ratio,Entities pValue,Entities FDR,#Reactions found,#Reactions total,Reactions ratio,Species identifier,Species name,Submitted entities found,Mapped entities,Found reaction identifiers
0,R-HSA-5619063,Defective SLC29A3 causes histiocytosis-lymphad...,8,8,0.000512,0.000002,0.002675,1,1,0.000067,9606,Homo sapiens,17596;16708;16750;17562;17748;16335;16704;17964,NaN,R-HSA-5628807
1,R-HSA-444209,Free fatty acid receptors,14,29,0.001855,0.000003,0.002675,5,5,0.000333,9606,Homo sapiens,16196;15366;30805;28875;28716;42504;28661;2836...,NaN,R-HSA-3296304;R-HSA-444047;R-HSA-444191;R-HSA-...
2,R-HSA-400508,"Incretin synthesis, secretion, and inactivation",18,51,0.003263,0.000009,0.005368,10,15,0.001000,9606,Homo sapiens,25858;16196;71466;15355;30805;28875;28716;4250...,NaN,R-HSA-381799;R-HSA-400500;R-HSA-381798;R-HSA-3...
3,R-HSA-381771,"Synthesis, secretion, and inactivation of Gluc...",17,47,0.003007,0.000011,0.005368,7,8,0.000533,9606,Homo sapiens,25858;16196;71466;15355;30805;28875;28716;4250...,NaN,R-HSA-381799;R-HSA-381798;R-HSA-383313;R-HSA-9...
4,R-HSA-83936,Transport of nucleosides and free purine and p...,13,33,0.002111,0.000050,0.019176,11,16,0.001067,9606,Homo sapiens,17596;17368;16235;16750;17562;16040;16708;1774...,NaN,R-HSA-109536;R-HSA-109539;R-HSA-109538;R-HSA-7...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1766,R-HSA-73857,RNA Polymerase II Transcription,76,1728,0.110557,1.000000,1.000000,278,941,0.062733,9606,Homo sapiens,17685;16796;16390;16070;86507;2700;28610;89613...,NaN,R-HSA-8937097;R-HSA-9007759;R-HSA-8878220;R-HS...
1767,R-HSA-212436,Generic Transcription Pathway,72,1586,0.101472,1.000000,1.000000,226,880,0.058667,9606,Homo sapiens,17685;16796;16390;16070;86507;2700;28610;89613...,NaN,R-HSA-8937097;R-HSA-9007759;R-HSA-8878220;R-HS...
1768,R-HSA-168256,Immune System,141,2664,0.170441,1.000000,1.000000,432,1723,0.114867,9606,Homo sapiens,16196;17203;17164;16113;18132;17285;16750;1619...,NaN,R-HSA-741386;R-HSA-688136;R-HSA-688137;R-HSA-8...
1769,R-HSA-9824446,Viral Infection Pathways,42,1158,0.074088,1.000000,1.000000,110,752,0.050133,9606,Homo sapiens,17981;17685;17361;189726;16796;9349;21803;2867...,NaN,R-HSA-9830805;R-HSA-182279;R-HSA-182286;R-HSA-...


In [16]:
df[df['Pathway identifier'] == 'R-HSA-3642279']

,Pathway identifier,Pathway name,#Entities found,#Entities total,Entities ratio,Entities pValue,Entities FDR,#Reactions found,#Reactions total,Reactions ratio,Species identifier,Species name,Submitted entities found,Mapped entities,Found reaction identifiers
23,R-HSA-3642279,TGFBR2 MSI Frameshift Mutants in Cancer,2,2,0.000128,0.018169,0.84948,1,1,0.000067,9606,Homo sapiens,15611;21803,NaN,R-HSA-3642203


In [17]:
entities_mapping = dict(zip(df['Pathway identifier'], df.coverage))

for node in G.nodes:
    if node in entities_mapping:
        G.nodes[node]['coverage'] = entities_mapping[node]
    else:
        G.nodes[node]['coverage'] = 0  

AttributeError: 'DataFrame' object has no attribute 'coverage'